First look: ดูโครงสร้าง ช่วงเวลา หน่วย และค่าหาย ไม่วิเคราะห์

In [21]:
import requests
import pandas as pd
import truststore; truststore.inject_into_ssl()

URL = "http://air4thai.pcd.go.th/services/getNewAQI_JSON.php"
r = requests.get(URL, timeout=30)
r.status_code, r.headers.get("Content-Type")

(200, 'application/json; charset=UTF-8')

In [22]:
data = r.json()
print(type(data), list(data.keys()) if isinstance(data, dict) else len(data))
stations = data["stations"]
print(len(stations))
stations[0]

<class 'dict'> ['stations']
180


{'stationID': '119t',
 'nameTH': 'สวนสาธารณะธารา',
 'nameEN': 'Thara Public Park',
 'areaTH': 'ต.ปากน้ำ อ.เมือง, กระบี่',
 'areaEN': 'Pak Nam Subdistrict, Mueang District, Krabi',
 'stationType': 'GROUND',
 'lat': '8.0506237',
 'long': '98.9180489',
 'forecast': [],
 'AQILast': {'date': '2026-11-24',
  'time': '11:00',
  'PM25': {'color_id': '1', 'aqi': '12', 'value': '7.0'},
  'PM10': {'color_id': '0', 'aqi': '-1', 'value': '-1'},
  'O3': {'color_id': '1', 'aqi': '14', 'value': '19'},
  'CO': {'color_id': '1', 'aqi': '4', 'value': '0.70'},
  'NO2': {'color_id': '1', 'aqi': '0', 'value': '1'},
  'SO2': {'color_id': '1', 'aqi': '1', 'value': '2'},
  'AQI': {'color_id': '1', 'aqi': '14', 'param': 'O3'}}}

In [23]:
df_now = pd.json_normalize(stations)
print(df_now.shape)
df_now.columns.tolist()

(180, 32)


['stationID',
 'nameTH',
 'nameEN',
 'areaTH',
 'areaEN',
 'stationType',
 'lat',
 'long',
 'forecast',
 'AQILast.date',
 'AQILast.time',
 'AQILast.PM25.color_id',
 'AQILast.PM25.aqi',
 'AQILast.PM25.value',
 'AQILast.PM10.color_id',
 'AQILast.PM10.aqi',
 'AQILast.PM10.value',
 'AQILast.O3.color_id',
 'AQILast.O3.aqi',
 'AQILast.O3.value',
 'AQILast.CO.color_id',
 'AQILast.CO.aqi',
 'AQILast.CO.value',
 'AQILast.NO2.color_id',
 'AQILast.NO2.aqi',
 'AQILast.NO2.value',
 'AQILast.SO2.color_id',
 'AQILast.SO2.aqi',
 'AQILast.SO2.value',
 'AQILast.AQI.color_id',
 'AQILast.AQI.aqi',
 'AQILast.AQI.param']

### สรุปการสำรวจโครงสร้างข้อมูลปัจจุบันจาก Air4Thai

1. **หนึ่งแถวของ `df_now` คืออะไร:** ข้อมูลการตรวจวัดคุณภาพอากาศล่าสุดของ **1 สถานีตรวจวัด** ณ ช่วงเวลาที่รายงาน (มีทั้งหมด 180 สถานี)
2. **คอลัมน์ข้อมูลพื้นฐานและเวลา:** 
   - รหัสสถานี: `stationID`
   - ชื่อสถานี: `nameTH` (ภาษาไทย), `nameEN` (ภาษาอังกฤษ)
   - พิกัด: `lat` (ละติจูด), `long` (ลองจิจูด)
   - วันที่และเวลา: `AQILast.date`, `AQILast.time`
3. **คอลัมน์ค่า PM2.5 และหน่วย:** มีทั้ง 2 ค่า คือ
   - ค่าความเข้มข้น: `AQILast.PM25.value` (หน่วยเป็นไมโครกรัมต่อลูกบาศก์เมตร หรือ µg/m³)
   - ค่าดัชนีคุณภาพอากาศ: `AQILast.PM25.aqi` (ไม่มีหน่วย / เป็นระดับดัชนี AQI)
4. **ชนิดข้อมูลและค่าแปลก:** ค่าตัวเลขทั้งหมดถูกเก็บเป็น **ข้อความ (`object` / `str`)** (สังเกตจากมีเครื่องหมายคำพูด `'...'` ครอบตัวเลข เช่น `'6.8'`) และพบการใช้ **ค่าติดลบ (`'-1'`)** แทนค่าหายหรือค่าที่ตรวจวัดไม่ได้ (เช่นในข้อมูลสถานีแรกพบ `'PM10': {'aqi': '-1', 'value': '-1'}`)

In [24]:
hourly = pd.read_csv("../data/sample/pm25_hourly_sample.csv", parse_dates=["ts"])
st = pd.read_csv("../data/sample/stations_sample.csv")

# ไฟล์ตัวอย่างมีข้อมูลช่วง ก.ย. - พ.ย. 2026 จึงใช้ข้อมูลชุดนี้ในการสำรวจเบื้องต้น
explore = hourly.copy()

print(explore.shape, explore["station_id"].nunique(), "สถานี")
print(explore["ts"].min(), "→", explore["ts"].max())
print(explore["ts"].diff().value_counts().head(3))   # ความถี่ของข้อมูล (ควรเป็น 1 ชั่วโมง)
explore.isna().mean().round(3)

(333216, 8) 178 สถานี
2026-09-01 00:00:00 → 2026-11-17 23:00:00
ts
0 days 01:00:00       333038
-78 days +01:00:00       177
Name: count, dtype: int64


station_id    0.000
ts            0.000
pm25          0.071
pm10          0.646
o3            0.849
co            0.811
no2           0.794
so2           0.828
dtype: float64

In [25]:
(explore.groupby("station_id")["pm25"]
        .apply(lambda s: s.isna().mean())
        .sort_values(ascending=False)
        .head(10))

station_id
o67    0.986645
o29    0.942842
o59    0.933226
o64    0.923077
o68    0.905449
76t    0.803419
74t    0.748932
18t    0.705128
75t    0.704060
26t    0.618056
Name: pm25, dtype: float64

### สรุปสิ่งที่พบจากการสำรวจข้อมูลย้อนหลังเบื้องต้น (First Look)

1. **ขอบเขตและปริมาณข้อมูล:** พบว่าข้อมูลตัวอย่างประกอบด้วย 178 สถานี รวมทั้งหมด 333,216 แถว ครอบคลุมช่วงเวลาตั้งแต่วันที่ 1 กันยายน 2026 เวลา 00:00 น. ถึง 17 พฤศจิกายน 2026 เวลา 23:00 น.
2. **ความถี่ของข้อมูล:** พบว่าความถี่หลักของการบันทึกข้อมูลคือทุกๆ 1 ชั่วโมง (`0 days 01:00:00` จำนวน 333,038 รายการ) และพบการกระโดดข้ามเวลาติดลบ (`-78 days`) จำนวน 177 ครั้ง ซึ่งเกิดจากการเปลี่ยนสถานีตรวจวัดเมื่อเรียงข้อมูลต่อกัน
3. **สัดส่วนข้อมูลสูญหาย (Missing Values):** พบว่าคอลัมน์ `pm25` มีค่าหายเพียงประมาณ 7.1% (`0.071`) ในขณะที่ตัวแปรสารมลพิษอื่นมีสัดส่วนค่าหายสูงมาก เช่น `pm10` หายถึง 64.6%, `no2` หาย 79.4%, `co` หาย 81.1%, `so2` หาย 82.8% และ `o3` หายสูงสุดถึง 84.9% ควรตรวจสอบต่อว่าสถานีใดบ้างที่ขาดอุปกรณ์ตรวจวัดก๊าซเหล่านี้